# Pladebevægelser og transformationer

I dagens øvelse skal vi regne på effekten af pladebevægelser. Vi loader først de nødvendige libraries

In [ ]:
from math import sqrt
from numpy import array, zeros, set_printoptions
from pyproj import Transformer

set_printoptions(precision=4, suppress=True)  # print altid 4 decimaler

Indsæt nu jeres observations tidspunkt i decimal år. Dette kan udregnes ved 

Decimal år = Årstal + (dag_nummer_i_indeværende_år / 365). 

I kan vælge at bruge en online tabel til at finde jeres dag nummer i det indeværende år, eller i kan vælge at tælle. (obs : det går langsomt hvis i skal tælle.) 

I skal også indsætte jeres PPP løsninger og koordinater for fikspunktet.
 - fikspunkt_xyz er koordinaterne for det målte fikspunkt
 - fase_xyz er den syntetiske baselinieløsning i har lavet 
 - ppp_xyz er fra jeres NRCan PPP løsning

In [ ]:
gr96_time = 1996.623
# Hvad er jeres observations tid? (decimal år. I kan udregne på følgende måde : Decimal år = Årstal + (dag_nummer_på_året / 365)
obs_time = 1996 + (227/365) # <--- opdater med jeres observationstid her

fikspunkt_xyz = array([[1484594.3445, -2018196.7034, 5845881.9379, gr96_time],
                 [1484592.6769, -2018386.6226, 5845816.0950, gr96_time],
                 [1483844.4633, -2018816.6578, 5845827.0648, gr96_time], 
                 [1484675.6605, -2019004.2859, 5845594.2066, gr96_time]])

fase_xyz = array([[1484594.3037, -2018196.6384, 5845881.8085, gr96_time],
                  [1484592.5593, -2018386.6654, 5845816.2019, gr96_time],
                  [1483844.4253, -2018816.6197, 5845827.0390, gr96_time],
                  [1484675.7049, -2019004.3355, 5845594.3942, gr96_time]])

ppp_xyz = array([[1234567.8910, 1234567.8910, 1234567.8910, obs_time], #  <------ Indsæt PPP løsning her for punkt 1
                 [1234567.8910, 1234567.8910, 1234567.8910, obs_time], #  <------ Indsæt PPP løsning her for punkt 2
                 [1234567.8910, 1234567.8910, 1234567.8910, obs_time], #  <------ Indsæt PPP løsning her for punkt 3
                 [1234567.8910, 1234567.8910, 1234567.8910, obs_time]]) # <------ Indsæt PPP løsning her for punkt 4

## Afvigelse fra PPP løsning

Vi udregner nu afvigelsen ifht vores PPP løsning, uden nogen form for transformation

In [ ]:
# find differencen
diff_xyz = ppp_xyz - fikspunkt_xyz
print(diff_xyz)

# initialiser variable
sum_x = 0.0
sum_y = 0.0
sum_z = 0.0

# regn den samlede sum af differencen
for pos in range(0, len(ppp_xyz)):
    sum_x += diff_xyz[pos, 0]
    sum_y += diff_xyz[pos, 1]
    sum_z += diff_xyz[pos, 2]

# regn den gennemsnitlige difference
diff_x = sum_x / len(ppp_xyz)
diff_y = sum_y / len(ppp_xyz)
diff_z = sum_z / len(ppp_xyz)

print(f"X: {diff_x:.3f} Y: {diff_y:.3f} Z: {diff_z:.3f}")
print(f"Afvigelse: {sqrt(diff_x ** 2 + diff_y ** 2 + diff_z ** 2):.3f}")

## Kontinentalpladebevægelse tilbage 1996.623
GR96 er defineret i epoken 1996.623 (15. august, 1996), så koordinaterne skal derfor transformeres fra observationstidspunktet og til 1996.623.

Grønland og dermed GR96, er en del af den Nordamerikanske plade og vi skal dermed bruge bevægelsen af denne plade for at transformerer i tid.

Vi bruger *+inv* for at angive at vi transformerer tilbage i tid og *+init=ITRF2014:NOAM* for at angive at vi bruger bevægelsen af den Nordamerikanske plade som defineret, i ITRF2014.

In [ ]:
# flyt det Nord Amerikanske kontinent tilbage til 1996.623 (stadig i ITRF2014) 
pipeline = ("+ellps=GRS80 +proj=pipeline " +
            f"+step +inv +init=ITRF2014:NOAM +t_epoch={gr96_time}")
trans_gr96_t = Transformer.from_pipeline(pipeline)

ppp_xyz_t = zeros((4, len(ppp_xyz)))
for pos in range(0, len(ppp_xyz)):
    trans = array(trans_gr96_t.transform(*ppp_xyz[pos]))
    ppp_xyz_t[pos] = trans

ppp_xyz_t[:,3] = gr96_time

diff_xyz_t = ppp_xyz_t - fikspunkt_xyz
print(diff_xyz_t)

# initialiser variable
sum_x = 0.0
sum_y = 0.0
sum_z = 0.0

# regn den samlede sum af differencen
for pos in range(0, len(ppp_xyz)):
    sum_x += diff_xyz_t[pos, 0]
    sum_y += diff_xyz_t[pos, 1]
    sum_z += diff_xyz_t[pos, 2]

# regn den gennemsnitlige difference
diff_x = sum_x / len(ppp_xyz)
diff_y = sum_y / len(ppp_xyz)
diff_z = sum_z / len(ppp_xyz)

print(f"X: {diff_x:.3f} Y: {diff_y:.3f} Z: {diff_z:.3f}")
print(f"Afvigelse: {sqrt(diff_x ** 2 + diff_y ** 2 + diff_z ** 2):.3f}")

## Transformer fra ITRF2014 referenceramme til ITRF94
Til slut skal vi transformere fra ITRF2014 referencerammen til ITRF94 referencerammen, som er den GR96 er defineret i.

In [ ]:
# transformer fra ITRF2014 til ITRF94 i 1996.623
pipeline = ("+ellps=GRS80 +proj=pipeline " +
            f"+step +init=ITRF2014:ITRF94 +t_obs={gr96_time}")
trans_itrf = Transformer.from_pipeline(pipeline)

ppp_xyz_t_gr96 = zeros((4, len(ppp_xyz)))
for pos in range(0, len(ppp_xyz)):
    trans = array(trans_itrf.transform(*ppp_xyz_t[pos]))
    ppp_xyz_t_gr96[pos] = trans

diff_xyz_t_gr96 = ppp_xyz_t_gr96 - fikspunkt_xyz
print(diff_xyz_t_gr96)

sum_x = 0.0
sum_y = 0.0
sum_z = 0.0
for pos in range(0, len(ppp_xyz)):
    sum_x += diff_xyz_t_gr96[pos, 0]
    sum_y += diff_xyz_t_gr96[pos, 1]
    sum_z += diff_xyz_t_gr96[pos, 2]
    
diff_x = sum_x / len(ppp_xyz)
diff_y = sum_y / len(ppp_xyz)
diff_z = sum_z / len(ppp_xyz)

print(f"X: {diff_x:.3f} Y: {diff_y:.3f} Z: {diff_z:.3f}")
print(f"Afvigelse: {sqrt(diff_x ** 2 + diff_y ** 2 + diff_z ** 2):.3f}")


## Sammenlign med jeres syntetiske baselinie

Prøv til sidst at sammenligne jeres syntetiske baselinieløsning med de definerede koordinater fra fikspunktsbeskrivelserne.

In [ ]:
diff_xyz_base = fase_xyz - fikspunkt_xyz

sum_x = 0.0
sum_y = 0.0
sum_z = 0.0
for pos in range(0, len(ppp_xyz)):
    sum_x += diff_xyz_base[pos, 0]
    sum_y += diff_xyz_base[pos, 1]
    sum_z += diff_xyz_base[pos, 2]
    
diff_x = sum_x / len(ppp_xyz)
diff_y = sum_y / len(ppp_xyz)
diff_z = sum_z / len(ppp_xyz)

print(f"X: {diff_x:.3f} Y: {diff_y:.3f} Z: {diff_z:.3f}")
print(f"Afvigelse: {sqrt(diff_x ** 2 + diff_y ** 2 + diff_z ** 2):.3f}")

## Kommenter
- Hvorfor er det vigtigt at transformere i tid?
- Kan du komme på et eksempel, hvor det ville være nødvendigt at transformere i tid?